## ETL Bronze – Ingesta raw (WeatherAPI)

### Propósito
Ingerir datos crudos desde WeatherAPI y almacenarlos como JSON (sin transformaciones) en una estructura tipo data lake, particionada por `endpoint`, `city` y `date`. Cada archivo incluye el payload original + metadata de ingesta.

### Entrada
- Fuente: WeatherAPI (`http://api.weatherapi.com/v1`)
- Autenticación: widget de Databricks `api_key`
- Fecha de proceso:
  - `date_str`: `YYYY-MM-DD` (UTC, fecha del momento de ejecución)
  - `timestamp`: `YYYY-MM-DDTHH-MM-SSZ` (UTC, usado como nombre de archivo)

### Endpoints consultados (por ciudad)
- `forecast`: pronóstico a 3 días (`days=3`, incluye `aqi=yes&alerts=yes`)
- `history`: histórico del día `date_str` (`dt=date_str`)
- `astronomy`: datos astronómicos

### Ciudades
Lista hardcodeada en el notebook (ej.: Venado Tuerto, Rosario, Firmat, Rafaela, Casilda, Cañada de Gomez, San Lorenzo, El Trebol, San Justo, Barrancas).  
Antes de escribir, se normaliza el nombre (minúsculas, espacios a `_`, sin tildes).

### Salida (almacenamiento)
Ruta base (Volume):
- `/Volumes/workspace/default/bronce_clima`

Estructura de escritura:
- `/Volumes/workspace/default/bronce_clima/<endpoint>/city=<city_normalizada>/date=<YYYY-MM-DD>/<timestamp>.json`

Nota: al escribir en carpetas `city=...` y `date=...`, Spark puede leer esas particiones y materializarlas como columnas `city` y `date` al hacer `spark.read.json(...)`.

### Formato del JSON guardado
Cada archivo se guarda con esta estructura:
- `data`: respuesta original de WeatherAPI (JSON)
- `metadata`:
  - `ciudad`: ciudad original (no normalizada)
  - `endpoint`: nombre del endpoint (`forecast`/`history`/`astronomy`)
  - `ingestion_time`: `timestamp` UTC (string)
  - `source`: `"weatherapi"`

### Manejo de errores
- Si `status_code != 200`: se escribe un JSON de error en:
  - `/Volumes/workspace/default/bronce_clima/_errors/<timestamp>_<city>_<endpoint>.json`
  - Contiene: `ciudad`, `endpoint`, `status_code`, `timestamp`
- Si la API responde `200` pero incluye la clave `"error"`: se registra en logs y se salta ese caso.
- En excepciones: se escribe un JSON de error en `_errors` con `error` y se registra el stacktrace.

### Control de rate limit
- `sleep(1)` por request (después de cada llamada) para reducir riesgo de rate limit.


In [0]:
import requests
import json
import logging
import time
import unicodedata
from datetime import datetime

In [0]:
api_key = dbutils.widgets.get("api_key")
base_url = "http://api.weatherapi.com/v1"
path_raiz = "/Volumes/workspace/default/bronce_clima"

# LOGGING
logging.basicConfig(level=logging.INFO)

def normalizar_texto(texto):
    texto = texto.lower()
    texto = texto.replace(" ", "_")
    texto = unicodedata.normalize("NFD", texto)
    texto = "".join([c for c in texto if unicodedata.category(c) != 'Mn'])
    texto = texto.encode("ascii", "ignore").decode("utf-8")
    return texto


now = datetime.utcnow()
date_str = now.strftime("%Y-%m-%d")
timestamp = now.strftime("%Y-%m-%dT%H-%M-%SZ")

# ENDPOINTS
endpoints = {
    "forecast": lambda ciudad: f"{base_url}/forecast.json?key={api_key}&q={ciudad}&days=3&aqi=yes&alerts=yes",
    "history": lambda ciudad: f"{base_url}/history.json?key={api_key}&q={ciudad}&dt={date_str}",
    "astronomy": lambda ciudad: f"{base_url}/astronomy.json?key={api_key}&q={ciudad}"
}



ciudades = [
    "Venado Tuerto", "Rosario", "Firmat", "Rafaela",
    "Casilda", "Cañada de Gomez", "San Lorenzo",
    "El Trebol", "San Justo","Barrancas"
]

for ciudad in ciudades:
    city_folder = normalizar_texto(ciudad)

    for endpoint_name, url_func in endpoints.items():
        url = url_func(ciudad)

        try:
            response = requests.get(url)
            if response.status_code == 200:
                raw_data = response.json()

                if "error" in raw_data:
                    logging.error(f"API error en {endpoint_name} - {ciudad}: {raw_data}")
                    continue
                # METADATA
                enriched_data = {
                    "data": raw_data,
                    "metadata": {
                        "ciudad": ciudad,
                        "endpoint": endpoint_name,
                        "ingestion_time": timestamp,
                        "source": "weatherapi"
                    }
                }

                # PATH
                full_folder_path = f"{path_raiz}/{endpoint_name}/city={city_folder}/date={date_str}"
                file_name = f"{timestamp}.json"
                full_file_path = f"{full_folder_path}/{file_name}"

                # ESCRITURA EN VOLUMEN
                dbutils.fs.put(
                    full_file_path,
                    json.dumps(enriched_data, ensure_ascii=False),
                    overwrite=True
                )

                logging.info(f"Guardado OK: {endpoint_name} | {city_folder}")

            else:
                # ERROR API
                error_info = {
                    "ciudad": ciudad,
                    "endpoint": endpoint_name,
                    "status_code": response.status_code,
                    "timestamp": timestamp
                }

                error_path = f"{path_raiz}/_errors/{timestamp}_{city_folder}_{endpoint_name}.json"

                dbutils.fs.put(
                    error_path,
                    json.dumps(error_info),
                    overwrite=True
                )

                logging.error(f"Error {response.status_code} en {endpoint_name} - {ciudad}")

        except Exception as e:
            # ERROR CRÍTICO
            error_info = {
                "ciudad": ciudad,
                "endpoint": endpoint_name,
                "error": str(e),
                "timestamp": timestamp
            }

            error_path = f"{path_raiz}/_errors/{timestamp}_{city_folder}_{endpoint_name}_exception.json"

            dbutils.fs.put(
                error_path,
                json.dumps(error_info),
                overwrite=True
            )

            logging.exception(f"Fallo crítico en {endpoint_name} - {ciudad}")

        # RATE LIMIT
        time.sleep(1)